In [1]:
### Scientific Computing
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### IPython
from IPython.display import display

### Qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

### Qiskit IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, Executor, QuantumProgram
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options_models.noise_learner_v3_options import NoiseLearnerV3Options
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

### Samplomatic
import samplomatic
from samplomatic import Twirl, InjectNoise, ChangeBasis, build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions, get_annotation

### Qiskit Addons
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.noise_management import trex_factors, gamma_from_noisy_boxes
from qiskit_addon_pna import generate_noise_mitigating_observable
from qiskit_addon_slc.bounds import compute_backward_bounds, compute_forward_bounds, compute_local_scales, merge_bounds
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone

ModuleNotFoundError: No module named 'qiskit_ibm_runtime'

## 1 Putting it together: the 1D Ising chain

Up until now, we have been using a 2-qubit toy CZ circuit to introduce each Samplomatic tool and understand the overall workflow. From here on, we work with a 1D transverse-field Ising chain. This is the same system Chapter 3 continues with. 

The boxing → build → noise-learning → execute workflow is the same and the circuit is now deep enough for per-layer rates to compound into a measurable deviation. 

There are two exercises at the end of this section that put your knowledge of everything we have covered in Chapter 2 to the test. Exercise 2 asks you to box a deeper Ising mirror so `NoiseLearnerV3` has more layers to characterize, and Exercise 3 assembles the resulting program — boxed circuit, learned noise, samplex, `QuantumProgram` — into the form the `Executor` consumes.

### 1. 1  Hamiltonian and Trotter circuit

The system under consideration is the __1D transverse-field Ising chain__, with the following Hamiltonian:

$$H \;=\; -J \sum_{\langle i,j\rangle} Z_i Z_j \;+\; h \sum_i X_i.$$

This Hamiltonian is a standard utility-scale benchmark; the same Hamiltonian (on a 2D heavy-hex lattice) drove IBM's 127-qubit utility demonstration, [Kim et al., *Evidence for the utility of quantum computing before fault tolerance*, Nature **618**, 500–505 (2023)](https://www.nature.com/articles/s41586-023-06096-3). We use a 1D chain because it makes the brickwork structure easy to inspect, and Chapter 3's PNA and SLC tutorials continue with the same system.

We define the helper function `construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=True)` below. This function builds each Ising circuit by applying $S^\dagger$ on both qubits followed by a $\mathrm{CZ}$. Because all three are diagonal and commute, the sequence equals $R_{ZZ}(-\pi/2)$ up to a global phase. Writing the circuit this way rather than as a parametric `RZZ` keeps every two-qubit gate a CZ, so the boxing and dressing machinery from sections 2.1–2.5 applies directly.

The circuit is a simple fixed-angle, kicked-Ising-style Trotter step. The transverse-field rotation is controlled by `rx_angle` and the ZZ-interaction is realized by a fixed CZ-based block. We won't go into any more detail here about the physics of the Ising circuit as the focus of this lab is on noise learning and error mitigation and the Ising circuit simply provides us with a nice example to work with. 

In [ ]:
def construct_ising_circuit(num_qubits: int,
                            num_trotter_steps: int,
                            rx_angle: float,
                            barrier: bool = True) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    for _ in range(num_trotter_steps):
        qc.rx(rx_angle, range(num_qubits))
        if barrier:
            qc.barrier()
        for first_qubit in (1, 2):
            for idx in range(first_qubit, num_qubits, 2):
                qc.sdg([idx - 1, idx])
                qc.cz(idx - 1, idx)
        if barrier:
            qc.barrier()
    return qc

ising = construct_ising_circuit(num_qubits=4, num_trotter_steps=1, rx_angle=np.pi / 8)
ising.draw("mpl", fold=-1)

### 1.2  The mirror trick

In Chapter 3, we will be comparing _unmitigated_ results to _error mitigated_ results for the 1D Ising chain circuit. In order to verify the effect of the error mitigation techniques implemented, we need to know the ideal result that we are aiming for. For an arbitrary circuit, this would require a separate classical simulation, and at utility scale that simulation is itself expensive or even untractable. This is where the mirror trick comes in.

The mirror trick takes the circuit of interest and appends its inverse onto the end of the circuit, i.e. we are applying the circuit forwards and then backwards. For an ideal quantum computer with no noise, the quantum state will return to the initial state: $|0\rangle^{\otimes N}$. A useful consequence of this is that we can easily calculate the ideal expectation value for any observable without having to perform any expensive classical simulations of the circuit. For example,  we know the ideal expectation value of $Z$ on every qubit is 1, i.e. $\langle Z_i \rangle = +1$, for all $i$. Any deviation of the measured $\langle Z_i \rangle$ from $+1$ is due hardware noise. That is what makes the mirror a standard benchmark for error mitigation.

In Qiskit, this is implemented using `ising.compose(ising.inverse())`, appending $U^\dagger$ to $U$. With Qiskit's right-to-left convention this realizes $U_\text{mirror} = U^\dagger U = I$. 

In [ ]:
mirror = ising.compose(ising.inverse())
mirror.measure_all()
mirror_isa = isa_pm.run(mirror)
mirror_isa.draw("mpl", fold=-1, idle_wires=False, scale=0.5)

### 1.3  Boxing the Ising mirror

We have the circuit and the workflow; the next step is to put them together. 

Below we convert the mirrored circuit into the form that `NoiseLearnerV3` consumes. We reuse `noise_learning_boxing_pm` from section 2.3.1 unchanged: both circuits are built from CZ entangling gates, so the same boxing strategy applies and each gate layer ends up wrapped in a `Twirl` + `InjectNoise` box.

`find_unique_box_instructions(...)` then collapses the boxed circuit to its *structurally distinct* layers, remember that equivalent boxes share one noise model so `NoiseLearnerV3` only needs to characterize each unique layer once. 
- `undress_boxes=True` strips the random-Pauli dressing before comparison to find unique layers, since dressing differs between boxes but the underlying layer does not. 
- `normalize_annotations=None` keeps only `Twirl` and `InjectNoise` box annotations and discards everything else


In [ ]:
boxed_circuit_ising = noise_learning_boxing_pm.run(mirror_isa)
unique_layers_ising = find_unique_box_instructions(
    boxed_circuit_ising,
    normalize_annotations=None,
    undress_boxes=True,
)

print(f"Unique 2Q-gate layers: {len(unique_layers_ising)}")
for i, inst in enumerate(unique_layers_ising):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a is not None else "(none)"
    print(f"  layer {i}: ref = {ref}")

boxed_circuit_ising.draw('mpl', idle_wires=False)


With the unique layers extracted, we inspect them: their `ref` strings (which `NoiseLearnerV3` will report back under) and the gate content of each. We see that even though the Ising circuit has 5 boxes, there are only 3 unique layers, one of which is the measurement layer.

In [ ]:

for i, inst in enumerate(unique_layers_ising):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a else "(none)"
    print(f"Layer {i}: ref = {ref}")
    display(inst.operation.body.draw("mpl", fold=-1, idle_wires=False))

### 1.4  Learning the noise

The boxed circuit is ready; now we can learn its noise. We pass `unique_layers_ising` to `NoiseLearnerV3` and assemble the result into `refs_to_noise_models_ising`, a dictionary mapping each layer's `InjectNoise.ref` to the learned `PauliLindbladMap`. Once both gate layers are characterized, we can read off how different their noise profiles are.

_Note:_ The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding.After first execution, we recommend pasting the job id into the `NOISE_LEARN_JOB_ID_ISING` parameter and setting  `SUBMIT_NOISE_JOB_ISING = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 10 seconds (tested on ibm_fez)._ The usage estimate reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.

In [ ]:
NOISE_LEARN_JOB_ID_ISING = None # paste job_id here on re-run
SUBMIT_NOISE_JOB_ISING   = False   # set True to submit a fresh learning job
learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_ISING is not None:
    learner_job_ising = service.job(NOISE_LEARN_JOB_ID_ISING)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_ISING}")
elif SUBMIT_NOISE_JOB_ISING:
    learner_job_ising = learner.run(unique_layers_ising)
    NOISE_LEARN_JOB_ID_ISING = learner_job_ising.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_ISING}")
else:
    print("Set SUBMIT_NOISE_JOB_ISING=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_ISING and re-run.")


In [ ]:
learner_job_ising = service.job(NOISE_LEARN_JOB_ID_ISING)
print(f"{NOISE_LEARN_JOB_ID_ISING}  (status: {learner_job_ising.status()})")

In [ ]:
if learner_job_ising.status() == "DONE":
    noise_learner_result_ising = learner_job_ising.result()
else:
    print(f"Not done yet (status={learner_job_ising.status()}). Re-run cell when DONE.")

In [ ]:
if 'noise_learner_result_ising' in dir() and noise_learner_result_ising is not None:
    refs_to_noise_models_ising = noise_learner_result_ising.to_dict(unique_layers_ising, require_refs=False)
    print(f"refs_to_noise_models_ising has {len(refs_to_noise_models_ising)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


The dict is ready: each entry is one learned layer. To see what was learned, the cell below prints the two layers properties. For each layer, we print  the number of generators, how many of those carry a nonzero rate, the largest generator rate, the sum of all the rates in the layer (total noise), and which Pauli term contributes the most.

In [ ]:
if "refs_to_noise_models_ising" not in globals() or not refs_to_noise_models_ising:
    print("Fetch a completed Ising noise-learning result first.")
else:
    print("Layer comparison:\n")
    print(f"  {'ref':<10}{'#gens':>8}{'#nonzero':>10}{'max rate':>14}{'sum rates':>14}{'top generator':>22}")
    print("  " + "─" * 78)
    for ref, plm in refs_to_noise_models_ising.items():
        gens = plm.to_sparse_list()
        rates = [abs(r) for _, _, r in gens]
        nonzero = [r for r in rates if r > 0]
        top = max(gens, key=lambda g: abs(g[2]))
        top_label = f"{top[0]}@{tuple(top[1])} ({top[2]:.2e})"
        print(f"  {ref:<10}{len(gens):>8}{len(nonzero):>10}"
              f"{max(rates):>14.4e}{sum(rates):>14.4e}"
              f"{top_label:>22}")


While the two unique layers play similar roles in the brickwork — they implement the even bonds (here the layer with two CZ gates) and odd bonds (here the layer with just one CZ gate). The table above shows that their learned noise is different. 

The `sum rates` column gives each layer's total noise; the `top generator` column shows which channel dominates, and the dominant channel of one layer can be weak or absent in the other; the `#nonzero` column shows that one layer can concentrate its noise on fewer channels while the other spreads it across more.

This per-layer noise difference is the reason we want to implement error mitigation per layer. A uniform correction tuned for one layer's dominant generator would miss or over-correct the other's. This is precisely the gap that Runtime's whole-circuit `EstimatorOptions` (section 1.1) cannot close.

We will see in Chapter 3 how PNA absorbs each layer's *own* inverse channel into a noise mitigating observable and how SLC scales each layer along its own lightcone. Both methods read off the per-layer rates we just printed.

_Note:_ Your specific results will vary depending on the QPU backend you are using and the noise learned from the QPU. Noise drifts and calibrations happen daily. Even for the same QPU, the noise will change over time.

### Exercise 2 — Box a deeper Ising mirror

<div class="alert alert-block alert-success">

Sections 2.6.1–2.6.4 walked through the full noise learning workflow on a 4-qubit, 1-Trotter-step Ising circuit. 

Run the same workflow yourself on a deeper circuit. Exercise 3 will then take what you build here and assemble it into an `Executor` program.

- **Circuit:** 6-qubit Ising chain, 2 Trotter steps, `rx_angle = π/8`, `barrier=False`
- **Mirror:** compose `ising_ex2`, add a `barrier()`, compose `ising_ex2.inverse()`, then `measure_all()`
- **Transpile:** through `isa_pm`
- **Box:** through `noise_learning_boxing_pm`

The `barrier()` between the forward and inverse halves prevents the transpiler's optimization passes from cancelling the mirror into the identity.

Fill in the cell below so that `boxed_circuit_ex2` is a Twirl + InjectNoise–annotated boxed mirror circuit that `build()` accepts.

</div>

In [ ]:
ising_ex2 = None
mirror_ex2 = None
boxed_circuit_ex2 = None

#TODO: Your code below.

# 1. Build the 6-qubit, 2-step Ising circuit (rx_angle = π/8, barrier=False)
ising_ex2 = ...

# 2. Build mirror_ex2: forward + barrier + inverse + measure_all
#    (The barrier keeps the transpiler from cancelling the mirror.)
mirror_ex2 = QuantumCircuit(num_qubits)
mirror_ex2.compose(..., inplace=True)   # forward
mirror_ex2.barrier()
mirror_ex2.compose(..., inplace=True)   # inverse
mirror_ex2.measure_all()

# 3. Transpile through isa_pm and box through noise_learning_boxing_pm
mirror_ex2_isa = ...
boxed_circuit_ex2 = ...

In Exercise 2 you produced `boxed_circuit_ex2`, the boxed Ising mirror circuit with an `InjectNoise` slot on every gate layer. This boxed circuit is now ready for characterization. 

Before Exercise 3 assembles it into a runnable program, we fill those slots: this is the same `NoiseLearnerV3` workflow as in section 2.6.4, now on the deeper circuit. We run it the exact same way: submit the job, poll its status, then fetch the result. From the learned noise, we produce `refs_to_noise_models_ex3`, a `{ref: PauliLindbladMap}` dict that we need for Exercise 3. 

#### Learn the noise (necessary for Exercise 3):

_Note:_ The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding. After first execution, we recommend pasting the job id into the `NOISE_LEARN_JOB_ID_EX3` parameter and setting `SUBMIT_NOISE_JOB_EX3 = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 4 seconds (tested on ibm_fez)._ The usage estimate reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.

In [ ]:
unique_layers_ex3 = find_unique_box_instructions(
    boxed_circuit_ex2,
    normalize_annotations=None,
    undress_boxes=True,
)

NOISE_LEARN_JOB_ID_EX3 = None   # paste a saved job id to re-fetch
SUBMIT_NOISE_JOB_EX3   = False              # set True to submit a fresh job
learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_EX3 is not None:
    learner_job_ex3 = service.job(NOISE_LEARN_JOB_ID_EX3)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_EX3}")
elif SUBMIT_NOISE_JOB_EX3:
    learner_job_ex3 = learner.run(unique_layers_ex3)
    NOISE_LEARN_JOB_ID_EX3 = learner_job_ex3.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_EX3}")
else:
    print("Set SUBMIT_NOISE_JOB_EX3=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_EX3 and re-run.")

In [ ]:
learner_job_ex3 = service.job(NOISE_LEARN_JOB_ID_EX3)
status = learner_job_ex3.status()

print(f"job_id : {NOISE_LEARN_JOB_ID_EX3}")
print(f"status : {status}")

if status == "DONE":
    print("\n  Ready — proceed to Exercise 3.")

Once the job is `DONE`, fetch the result by running the cell below and collect it into `refs_to_noise_models_ex3`: the `{ref: PauliLindbladMap}` dict. one learned model per layer. We need this object for Exercise 3.

In [ ]:
noise_result_ex3 = learner_job_ex3.result()
refs_to_noise_models_ex3 = noise_result_ex3.to_dict(
    unique_layers_ex3, require_refs=False
)
print(f"Learned noise for {len(refs_to_noise_models_ex3)} layers: "
      f"{list(refs_to_noise_models_ex3.keys())}")

The learned noise models can be understood more clearly as a figure than as raw rates. The plot below shows each layer's 15 largest noise generators, with 1-body (single-qubit) and 2-body (two-qubit) terms colored separately. You can clearly see which generators dominant each layers noise model. 

In [ ]:
# plot the top 15 noise generators per layer
fig, axes = plt.subplots(1, len(refs_to_noise_models_ex3), figsize=(12, 4))
if len(refs_to_noise_models_ex3) == 1: axes = [axes]

for ax, (ref, plm) in zip(axes, refs_to_noise_models_ex3.items()):
    gens = sorted(plm.to_sparse_list(), key=lambda g: -abs(g[2]))[:15]
    labels = [f"{p}@{tuple(q)}" for p, q, _ in gens]
    rates  = [r for _, _, r in gens]
    colors = ['#2c7fb8' if len(p) == 1 else '#d95f0e' for p, _, _ in gens]
    ax.barh(range(len(labels)), rates, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Rate")
    ax.set_title(f"Layer {ref}")
    ax.grid(axis="x", alpha=0.3)

fig.legend(handles=[Patch(facecolor="#2c7fb8", label="1-qubit"),
                    Patch(facecolor="#d95f0e", label="2-qubit")],
           loc="upper right", fontsize=9)
fig.suptitle("Top 15 noise generators per layer")
plt.tight_layout()
plt.show()

Now we have the noise model, in particular the dict `noise_result_ex3`, we can run an Executor job. 

In, Exercise 3, we ask you to prepare the Executor job using `noise_result_ex3` and the `build()` function to prepare the template and samplex using the boxed circuit you prepared in Exercise 2. 

### Exercise 3 — Assemble an `Executor` program

<div class="alert alert-block alert-success">

The noise learning job above learned `refs_to_noise_models_ex3`, the `{ref: PauliLindbladMap}` dict with one noise model per layer of `boxed_circuit_ex2`.

This exercise plugs that noise model into the data structure the `Executor` runs, using all three tools introduced in Chapter 2 in one pass: the **Samplomatic** `build`, the **NoiseLearnerV3** result you just produced, and the **Executor**'s `QuantumProgram`.

Build the program in four steps:

- **Build:** call `build(boxed_circuit_ex2)` to get `template_ex3` and `samplex_ex3`
- **Inspect:** `samplex_ex3.inputs()` reports which `pauli_lindblad_maps.<ref>` slots the samplex expects — these refs must match the keys of `refs_to_noise_models_ex3`
- **Bind:** from `samplex_ex3.inputs()`, call `make_broadcastable()` then `bind(pauli_lindblad_maps=...)` with the learned dict → `samplex_args_ex3`
- **Assemble:** create a `QuantumProgram(shots=64)` and `append_samplex_item(...)` the template, samplex, and arguments → `program_ex3`

The grader checks that the program is assembled correctly — it does not run on hardware. An optional cell afterwards shows the expectation values this program returns from a real backend run.

</div>

In [ ]:
#TODO: Add Your code here and set the missing values.

template_ex3      = None
samplex_ex3       = None
samplex_args_ex3  = None
program_ex3       = None


# 1. Build the template and samplex from boxed_circuit_ex2
template_ex3, samplex_ex3 = ...

# 2. Inspect what the samplex expects (look at the pauli_lindblad_maps.<ref> slots)
print(samplex_ex3.inputs())

# 3. Bind the learned noise models into the samplex inputs.
#    Start from samplex_ex3.inputs(), make it broadcastable, then bind the
#    refs_to_noise_models_ex3 dict to the `pauli_lindblad_maps` argument.
samplex_args_ex3 = (
    samplex_ex3.inputs()
    .make_broadcastable()
    .bind(pauli_lindblad_maps=...)
)

# 4. Assemble the QuantumProgram with a single samplex item
program_ex3 = QuantumProgram(shots=64)
program_ex3.append_samplex_item(...)

#### (Optional) Running the program on hardware

This step is optional and not required to proceed to Chapter 3. It shows what `program_ex3` returns when actually run on hardward. This section follows the same pattern as section 2.5, where we submitted an `Executor` job for the toy circuit.

The circuit here is the Ising mirror circuit, so the ideal result is $\langle Z_i\rangle = +1$ on every qubit. Any deviation you see from this is due to noise the boxed-and-learned pipeline still leaves. In Chapter 3, we will implement error mitigation techniques to improve the results.

_Note:_ The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding. After first execution, we recommend pasting the job id into the `EXECUTOR_JOB_ID_EX3` parameter and setting `SUBMIT_EXECUTOR_JOB_EX3 = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 2 seconds (tested on ibm_pittsburgh)_. The usage estimate above reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.


In [ ]:
executor = Executor(backend)

EXECUTOR_JOB_ID_EX3      = None      # paste an Executor job_id here on re-run
SUBMIT_EXECUTOR_JOB_EX3  = False     # set True to submit a fresh Executor job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_EX3 is not None:
    exec_job_ex3 = service.job(EXECUTOR_JOB_ID_EX3)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_EX3}")
elif SUBMIT_EXECUTOR_JOB_EX3:
    exec_job_ex3 = executor.run(program_ex3)
    EXECUTOR_JOB_ID_EX3 = exec_job_ex3.job_id()
    print(f"Submitted Executor job: {EXECUTOR_JOB_ID_EX3}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_EX3=True to submit a fresh Executor job, "
          "or paste a saved job id into EXECUTOR_JOB_ID_EX3 and re-run.")

In [ ]:
exec_job_ex3 = service.job(EXECUTOR_JOB_ID_EX3)
print(f"{EXECUTOR_JOB_ID_EX3}  (status: {exec_job_ex3.status()})")

Plot the results:

In [ ]:
if exec_job_ex3.status() == "DONE":
    exec_result_ex3 = exec_job_ex3.result()
    data  = exec_result_ex3[0]
    meas  = data["meas"]
    flips = data["measurement_flips.meas"]

    corrected = np.bitwise_xor(meas, flips)
    z_vals    = 1 - 2 * corrected
    z_means   = z_vals.reshape(-1, z_vals.shape[-1]).mean(axis=0)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(z_means)), z_means, color="#2c7fb8")
    ax.axhline(1.0, color="gray", ls="--", lw=1, label="ideal ⟨Z⟩ = +1")
    ax.set_xlabel("Qubit")
    ax.set_ylabel("⟨Z⟩")
    ax.set_ylim(0, 1.05)
    ax.set_title("Mirror expectation values (boxed + learned, pre-mitigation)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"mean ⟨Z⟩ = {z_means.mean():+.4f}  (ideal = +1)")
else:
    print(f"Not done yet (status={exec_job_ex3.status()}). Re-run cell when DONE.")

#### Wrapping up Chapter 2:

Congratulations! You have completed Chapter 2 of Lab 3. In this chapter we introduced all the tools needed to implement the advanced error mitigation techniques of chapter 3. 

We walked through the entire Samplomatic + noise learning workflow first for a simple 2-qubit toy model and then for the more complex 4-qubit Ising chain and finally for the 6-qubit Ising chain. For each circuit, the workflow remained the same and only the circuits changed. 

- In __Exercise 2__ you boxed the 6-qubit Ising mirror circuit and the noise-learning step characterized its layers. 
- In __Exercise 3__, you assembled the boxed circuit, the previously learned noise, and the samplex into the `QuantumProgram` the `Executor` runs. 

Along the way we inspected the per-layer noise structure: we learnt that the `ref` strings attached to every layer with an `InjectNoise` annotation are the labels that we attach the learned noise information to. 

We saw that for the Ising mirror circuit, even though the layers play similar roles in the physics of the circuit, the learned noise model in each layer differs (in particular the layers have different dominant generators). This motivates exactly why we want error mitigation to act __per layer__. 

#### What is missing — and why Chapter 3 exists:

Importantly, the pipeline we have seen in chapter 2 measures noise and prepares it for execution but it does not remove it. So far, we have seen how twirling reshapes coherent error into a stochastic Pauli channel but does not remove it (it applies a finite number of randomization samples without ever applying the inverse noise). Also, the learned-noise dict describes the noise per layer, but again Chapter 2 never uses its inverse — under `inject_noise_strategy="no_modification"` the dict is a passthrough, which is why the Ising mirror circuit's results for $\langle Z_i\rangle$ sit below the ideal $+1$. As circuits deepen, per-layer rates compound to the point where unmitigated bias makes results untrustworthy and we need per-layer error mitigation.

Chapter 3 introduces techniques for the next step in which we will need to change the `inject_noise_strategy`. 
- PNA (`inject_noise_strategy=uniform_modification`) absorbs the inverse of the learned channel into a _noise mitigating observable_
- SLC (`inject_noise_strategy=individual_modification`) samples anti-noise along each observable's _lightcone_